# Plant Disease Classification using Transfer Learning

**Problem:** Automatic identification of plant diseases from leaf images.
**Dataset:** PlantVillage (colour leaf images, 38 classes of healthy and diseased crop leaves)
**Approach:** Comparison of a Basic CNN trained from scratch against a pretrained MobileNetV2 (Transfer Learning)
**Tools:** Python, TensorFlow / Keras, scikit-learn, Matplotlib, Seaborn, Gradio

---

## 1. Introduction

Plant diseases reduce crop yield and quality, and early detection is important for farmers. Manual inspection needs expert knowledge, so an image-based classifier can support quick and low-cost diagnosis.

This project builds an image-classification system that takes a leaf photo and predicts the crop and the disease (or healthy status). Two approaches are implemented and compared:

1. **Basic CNN:** a convolutional network trained from scratch.
2. **MobileNetV2 (Transfer Learning):** a network pretrained on ImageNet, adapted to leaf images using feature extraction followed by fine-tuning.

Both models are evaluated on the same unseen test set using accuracy, precision, recall, F1-score and confusion matrices.

## 2. Project Workflow

| Section | Description |
|---------|-------------|
| 3 | Environment setup |
| 4 | Dataset description |
| 5 | Exploratory data analysis |
| 6 | Train / validation / test split |
| 7 | Data loading and preprocessing |
| 8 | Model 1: Basic CNN |
| 9 | Model 2: Transfer Learning (MobileNetV2) |
| 10 | Evaluation metrics |
| 11 | Confusion matrices |
| 12 | Model comparison and best model |
| 13 | Demo application (Gradio) |
| 14 | Conclusion and future work |

*Note: the notebook is designed to run on Google Colab with a T4 GPU runtime.*

## 3. Environment Setup

Checking the GPU runtime, installing the required libraries and fixing random seeds for reproducible results.

In [ ]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("GPUs found:", tf.config.list_physical_devices('GPU'))

In [ ]:
# Install helper libraries (kagglehub for the dataset, gradio for the demo)
!pip install -q kagglehub gradio

In [ ]:
import os, random, shutil, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix)
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings("ignore")

# Fix the random seeds so results are repeatable
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("All libraries imported successfully")

## 4. Dataset Description

**PlantVillage** is a public benchmark dataset of about 54,000 colour photographs of plant leaves, covering 14 crop species (such as Tomato, Potato, Apple, Grape and Corn) and 38 classes of healthy and diseased leaves. The images are captured against a plain background, which makes the dataset suitable for studying disease patterns such as spots, discolouration and texture changes.

**Why this dataset:** it is a real-world problem, it is large and well labelled, and it has enough classes to make the comparison between a scratch-built CNN and a pretrained model meaningful.

The dataset is downloaded from Kaggle through `kagglehub`. The `color` version of the images is used.

In [ ]:
import kagglehub

path = kagglehub.dataset_download("abdallahalidev/plantvillage-dataset")
print("Downloaded to:", path)

# Show the top-level folder structure
for root, dirs, files in os.walk(path):
    depth = root.replace(path, "").count(os.sep)
    if depth <= 2:
        print("  " * depth + os.path.basename(root) + "/")

In [ ]:
# The dataset has three versions: color, grayscale, segmented. The 'color' images are used.
DATA_DIR = None
for root, dirs, files in os.walk(path):
    if os.path.basename(root).lower() == "color":
        DATA_DIR = root
        break

assert DATA_DIR is not None, "Could not find the 'color' folder. Check the folder structure printed above."
print("Using images from:", DATA_DIR)

## 5. Exploratory Data Analysis

The class distribution and sample images are inspected before modelling.

In [ ]:
classes = sorted([d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))])
counts = {c: len(os.listdir(os.path.join(DATA_DIR, c))) for c in classes}

print("Number of classes:", len(classes))
print("Total images:", sum(counts.values()))
print("Smallest class:", min(counts, key=counts.get), "->", min(counts.values()), "images")
print("Largest class :", max(counts, key=counts.get), "->", max(counts.values()), "images")

print("\nAll classes:")
for i, c in enumerate(classes):
    print(f"{i:2d}. {c}  ({counts[c]} images)")

In [ ]:
# Bar chart: number of images in each class
plt.figure(figsize=(14, 6))
plt.bar(range(len(classes)), [counts[c] for c in classes], color="seagreen")
plt.xticks(range(len(classes)), classes, rotation=90, fontsize=7)
plt.ylabel("Number of images")
plt.title("Images per class (original dataset)")
plt.tight_layout()
plt.show()

In [ ]:
# Show a few random sample images
plt.figure(figsize=(14, 6))
for i in range(10):
    c = random.choice(classes)
    img_name = random.choice(os.listdir(os.path.join(DATA_DIR, c)))
    img = Image.open(os.path.join(DATA_DIR, c, img_name))
    plt.subplot(2, 5, i + 1)
    plt.imshow(img)
    plt.title(c.replace("___", "\n").replace("_", " "), fontsize=8)
    plt.axis("off")
plt.suptitle("Sample images from the dataset")
plt.tight_layout()
plt.show()
print("Original image size:", img.size)

**Observations**
- The dataset contains 38 classes covering healthy and diseased leaves of several crops.
- The classes are **imbalanced**: some classes have thousands of images while others have only a few hundred.
- Images are colour photographs with a plain background, so visible disease symptoms (spots, colour changes, texture) are the main distinguishing features.

## 6. Train / Validation / Test Split

The images are split class by class (stratified) so that every class appears in all three subsets in the same proportion.

| Subset | Share | Purpose |
|--------|-------|---------|
| Train | 70% | Learning the model weights |
| Validation | 15% | Monitoring training, early stopping and tuning |
| Test | 15% | Final unbiased evaluation on unseen images |

**Justification of the split:** a 70/15/15 split leaves enough data for learning while still keeping a test set large enough for a reliable final score. The test set is never used during training or tuning.

To keep the training time practical on a free Colab GPU, at most `MAX_PER_CLASS` images are used per class. This also reduces the class imbalance observed in the EDA.

In [ ]:
MAX_PER_CLASS = 300          # images used per class (increase for a larger, slower, more accurate run)
SPLIT_DIR = "/content/plant_split"

if os.path.exists(SPLIT_DIR):
    shutil.rmtree(SPLIT_DIR)

rng = random.Random(SEED)
split_counts = {"train": 0, "val": 0, "test": 0}

for c in classes:
    files = sorted(os.listdir(os.path.join(DATA_DIR, c)))
    rng.shuffle(files)
    files = files[:MAX_PER_CLASS]

    train_f, temp_f = train_test_split(files, test_size=0.30, random_state=SEED)
    val_f, test_f = train_test_split(temp_f, test_size=0.50, random_state=SEED)

    for split_name, split_files in [("train", train_f), ("val", val_f), ("test", test_f)]:
        out_dir = os.path.join(SPLIT_DIR, split_name, c)
        os.makedirs(out_dir, exist_ok=True)
        for f in split_files:
            shutil.copy2(os.path.join(DATA_DIR, c, f), os.path.join(out_dir, f))
        split_counts[split_name] += len(split_files)

print("Images in each split:", split_counts)
print("Total used:", sum(split_counts.values()))

## 7. Data Loading and Preprocessing

- All images are resized to **128 x 128** pixels.
- Images are processed in batches of **32**.
- Pixel scaling is done inside each model: `[0, 1]` for the Basic CNN and `[-1, 1]` for MobileNetV2, which is the input range MobileNetV2 was pretrained with.
- **Data augmentation** (random flips, rotation and zoom) is applied only during training to improve generalisation.
- The test set is not shuffled so that predictions stay aligned with the true labels.

In [ ]:
IMG_SIZE = (128, 128)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(SPLIT_DIR, "train"), image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode="int", shuffle=True, seed=SEED)

val_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(SPLIT_DIR, "val"), image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode="int", shuffle=False)

# shuffle=False for the test set keeps predictions aligned with the labels
test_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(SPLIT_DIR, "test"), image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode="int", shuffle=False)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)
print("Number of classes:", NUM_CLASSES)

# Speed up data loading
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)
test_ds = test_ds.cache().prefetch(AUTOTUNE)

In [ ]:
# Data augmentation block, used inside both models
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
], name="data_augmentation")

In [ ]:
# Helper function to plot training curves
def plot_history(history, title):
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    ax[0].plot(history.history["accuracy"], label="Train accuracy")
    ax[0].plot(history.history["val_accuracy"], label="Validation accuracy")
    ax[0].set_title(title + " - Accuracy")
    ax[0].set_xlabel("Epoch"); ax[0].legend()

    ax[1].plot(history.history["loss"], label="Train loss")
    ax[1].plot(history.history["val_loss"], label="Validation loss")
    ax[1].set_title(title + " - Loss")
    ax[1].set_xlabel("Epoch"); ax[1].legend()
    plt.tight_layout()
    plt.show()

# Early stopping: stop training when validation accuracy stops improving
def get_callbacks():
    return [keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3,
                                          restore_best_weights=True)]

## 8. Model 1: Basic CNN

A convolutional network trained from scratch with random initial weights.

**Architecture:** 3 convolution blocks (32, 64, 128 filters) with max pooling, global average pooling, a dense layer of 128 units with dropout (0.4), and a softmax output layer with 38 units.

**Training setup:** Adam optimizer (learning rate 1e-3), sparse categorical cross-entropy loss, up to 15 epochs with early stopping on validation accuracy.

In [ ]:
def build_basic_cnn():
    inputs = keras.Input(shape=IMG_SIZE + (3,))
    x = data_augmentation(inputs)
    x = layers.Rescaling(1.0 / 255)(x)          # scale pixels from 0-255 to 0-1

    x = layers.Conv2D(32, 3, activation="relu", padding="same")(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(64, 3, activation="relu", padding="same")(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(128, 3, activation="relu", padding="same")(x)
    x = layers.MaxPooling2D()(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.4)(x)                  # dropout reduces overfitting
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = keras.Model(inputs, outputs, name="Basic_CNN")
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model

cnn_model = build_basic_cnn()
cnn_model.summary()

In [ ]:
# Train the Basic CNN
start = time.time()
cnn_history = cnn_model.fit(train_ds, validation_data=val_ds, epochs=15,
                            callbacks=get_callbacks(), verbose=1)
cnn_train_time = time.time() - start
print(f"\nBasic CNN training time: {cnn_train_time/60:.1f} minutes")

In [ ]:
plot_history(cnn_history, "Basic CNN")

## 9. Model 2: Transfer Learning with MobileNetV2

MobileNetV2 pretrained on ImageNet (about 1.4 million images) is used as a feature extractor. Training is done in two phases:

- **Phase 1, feature extraction:** the pretrained base is frozen and only the new classification head is trained (learning rate 1e-3).
- **Phase 2, fine-tuning:** the last 30 layers of the base are unfrozen and trained with a very small learning rate (1e-5) to adapt the features to leaf images.

In [ ]:
def build_transfer_model():
    base_model = keras.applications.MobileNetV2(
        input_shape=IMG_SIZE + (3,), include_top=False, weights="imagenet")
    base_model.trainable = False                 # freeze the pretrained part

    inputs = keras.Input(shape=IMG_SIZE + (3,))
    x = data_augmentation(inputs)
    x = layers.Rescaling(1.0 / 127.5, offset=-1)(x)   # MobileNetV2 expects pixels in the range -1 to 1
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = keras.Model(inputs, outputs, name="MobileNetV2_Transfer")
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model, base_model

tl_model, base_model = build_transfer_model()
tl_model.summary()

In [ ]:
# Phase 1: train only the new top layers (fast)
start = time.time()
tl_history1 = tl_model.fit(train_ds, validation_data=val_ds, epochs=8,
                           callbacks=get_callbacks(), verbose=1)
tl_time1 = time.time() - start
print(f"\nPhase 1 time: {tl_time1/60:.1f} minutes")

In [ ]:
# Phase 2: fine-tuning. Unfreeze the last 30 layers of MobileNetV2 and use a very small learning rate.
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

tl_model.compile(optimizer=keras.optimizers.Adam(1e-5),
                 loss="sparse_categorical_crossentropy",
                 metrics=["accuracy"])

start = time.time()
tl_history2 = tl_model.fit(train_ds, validation_data=val_ds, epochs=5,
                           callbacks=get_callbacks(), verbose=1)
tl_time2 = time.time() - start
tl_train_time = tl_time1 + tl_time2
print(f"\nPhase 2 time: {tl_time2/60:.1f} minutes")
print(f"Total Transfer Learning training time: {tl_train_time/60:.1f} minutes")

In [ ]:
# Combine both phases into a single history for plotting
combined = {k: tl_history1.history[k] + tl_history2.history[k] for k in tl_history1.history}
class _H: pass
tl_history = _H(); tl_history.history = combined
plot_history(tl_history, "MobileNetV2 Transfer Learning")

## 10. Evaluation on the Test Set

Both models are evaluated on the unseen test set using:

- **Accuracy:** share of test images classified correctly.
- **Precision:** of the images predicted as a class, the share that truly belong to it.
- **Recall:** of the images that truly belong to a class, the share that were found.
- **F1-score:** harmonic mean of precision and recall.

Because this is a multi-class problem with 38 classes, the **weighted average** of precision, recall and F1-score is reported.

In [ ]:
# True labels of the test set (same order as predictions because shuffle=False)
y_true = np.concatenate([y.numpy() for _, y in test_ds])
print("Test images:", len(y_true))

results = {}

def evaluate_model(model, name, train_time):
    y_prob = model.predict(test_ds, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)

    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0)

    results[name] = {
        "Accuracy": acc, "Precision": prec, "Recall": rec, "F1-score": f1,
        "Training time (min)": train_time / 60,
        "Parameters": model.count_params(),
        "y_pred": y_pred,
    }
    print(f"===== {name} =====")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-score : {f1:.4f}\n")

evaluate_model(cnn_model, "Basic CNN", cnn_train_time)
evaluate_model(tl_model, "MobileNetV2 (Transfer Learning)", tl_train_time)

In [ ]:
# Detailed per-class report for each model
for name in results:
    print("=" * 20, name, "=" * 20)
    print(classification_report(y_true, results[name]["y_pred"],
                                target_names=class_names, zero_division=0))

## 11. Confusion Matrices

Rows represent the true class and columns the predicted class. A strong diagonal means correct predictions; off-diagonal cells show which classes are confused with each other.

In [ ]:
def plot_confusion(name):
    cm = confusion_matrix(y_true, results[name]["y_pred"])
    plt.figure(figsize=(16, 13))
    sns.heatmap(cm, cmap="Blues", xticklabels=class_names, yticklabels=class_names, cbar=True)
    plt.xlabel("Predicted class"); plt.ylabel("True class")
    plt.title("Confusion Matrix - " + name)
    plt.xticks(fontsize=6, rotation=90); plt.yticks(fontsize=6)
    plt.tight_layout()
    plt.show()

plot_confusion("Basic CNN")

In [ ]:
plot_confusion("MobileNetV2 (Transfer Learning)")

In [ ]:
# Most frequent misclassifications of the best model
best_name_tmp = max(results, key=lambda n: results[n]["F1-score"])
cm = confusion_matrix(y_true, results[best_name_tmp]["y_pred"])
np.fill_diagonal(cm, 0)                      # ignore correct predictions
pairs = [(cm[i, j], class_names[i], class_names[j])
         for i in range(NUM_CLASSES) for j in range(NUM_CLASSES) if cm[i, j] > 0]
pairs.sort(reverse=True)

print("Top mistakes of", best_name_tmp)
for count, true_c, pred_c in pairs[:8]:
    print(f"{count} images of '{true_c}' were predicted as '{pred_c}'")

## 12. Model Comparison and Best Model

In [ ]:
comparison = pd.DataFrame({
    name: {k: v for k, v in vals.items() if k != "y_pred"}
    for name, vals in results.items()
}).T

comparison["Accuracy"] = (comparison["Accuracy"] * 100).round(2)
comparison["Precision"] = (comparison["Precision"] * 100).round(2)
comparison["Recall"] = (comparison["Recall"] * 100).round(2)
comparison["F1-score"] = (comparison["F1-score"] * 100).round(2)
comparison["Training time (min)"] = comparison["Training time (min)"].astype(float).round(1)
comparison["Parameters"] = comparison["Parameters"].astype(int)
comparison

In [ ]:
# Bar chart comparison
metrics = ["Accuracy", "Precision", "Recall", "F1-score"]
x = np.arange(len(metrics))
width = 0.35

plt.figure(figsize=(9, 5))
for i, name in enumerate(comparison.index):
    plt.bar(x + i * width, [comparison.loc[name, m] for m in metrics], width, label=name)
plt.xticks(x + width / 2, metrics)
plt.ylabel("Score (%)")
plt.ylim(0, 105)
plt.title("Basic CNN vs Transfer Learning (Test set)")
plt.legend()
plt.show()

best_model_name = comparison["F1-score"].astype(float).idxmax()
print("Best-performing model:", best_model_name)

### Discussion

The analysis below is generated from the measured results of this run.

In [ ]:
cnn_r = results["Basic CNN"]
tl_r = results["MobileNetV2 (Transfer Learning)"]

acc_gain = (tl_r["Accuracy"] - cnn_r["Accuracy"]) * 100
f1_gain = (tl_r["F1-score"] - cnn_r["F1-score"]) * 100

print(f"Basic CNN        : accuracy {cnn_r['Accuracy']*100:.2f}%, F1-score {cnn_r['F1-score']*100:.2f}%, "
      f"training time {cnn_r['Training time (min)']:.1f} min, {cnn_r['Parameters']:,} parameters")
print(f"MobileNetV2 (TL) : accuracy {tl_r['Accuracy']*100:.2f}%, F1-score {tl_r['F1-score']*100:.2f}%, "
      f"training time {tl_r['Training time (min)']:.1f} min, {tl_r['Parameters']:,} parameters")
print()
print(f"Difference in accuracy (TL minus CNN): {acc_gain:+.2f} percentage points")
print(f"Difference in F1-score (TL minus CNN): {f1_gain:+.2f} percentage points")
print(f"Best-performing model: {best_model_name}")

**Interpretation.** A pretrained network such as MobileNetV2 has already learned general visual features (edges, textures, shapes and colours) from ImageNet, so it only needs to adapt these features to leaf images. The Basic CNN starts from random weights and has to learn all features from the limited number of leaf images available. This difference is the main reason for the gap in performance and training behaviour observed above. The trade-off is that the pretrained model is larger and expects a fixed input format, while the Basic CNN is smaller and simpler.

## 13. Demo Application

The best model is saved and wrapped in a Gradio web interface. A user uploads a leaf image and receives the top predicted classes with confidence scores.

In [ ]:
# Save both trained models
cnn_model.save("basic_cnn_plant.keras")
tl_model.save("mobilenetv2_plant.keras")

best_model = tl_model if "MobileNet" in best_model_name else cnn_model
print("Demo will use:", best_model_name)

In [ ]:
import gradio as gr

def pretty(name):
    return name.replace("___", " - ").replace("_", " ")

pretty_names = [pretty(c) for c in class_names]

def predict_leaf(image):
    if image is None:
        return None
    img = Image.fromarray(image).convert("RGB").resize(IMG_SIZE)
    arr = np.expand_dims(np.array(img, dtype="float32"), axis=0)
    probs = best_model.predict(arr, verbose=0)[0]
    return {pretty_names[i]: float(probs[i]) for i in range(NUM_CLASSES)}

demo = gr.Interface(
    fn=predict_leaf,
    inputs=gr.Image(label="Upload a plant leaf image"),
    outputs=gr.Label(num_top_classes=3, label="Prediction"),
    title="Plant Disease Detector",
    description="Upload a leaf photo to detect the plant disease. Model: MobileNetV2 Transfer Learning trained on PlantVillage.",
)
demo.launch(share=True, debug=False)

The demo launches with a public link. Screenshots of the interface with sample predictions are included in the project README.

## 14. Conclusion and Future Work

### Conclusion

In [ ]:
print("CONCLUSION")
print("-" * 60)
print(f"- A plant disease classifier for {NUM_CLASSES} classes was built on the PlantVillage dataset.")
print(f"- Images used: {sum(split_counts.values())} (train {split_counts['train']}, "
      f"validation {split_counts['val']}, test {split_counts['test']}).")
print(f"- Basic CNN test accuracy      : {cnn_r['Accuracy']*100:.2f}%")
print(f"- MobileNetV2 test accuracy    : {tl_r['Accuracy']*100:.2f}%")
print(f"- Best-performing model        : {best_model_name} "
      f"(weighted F1-score {results[best_model_name]['F1-score']*100:.2f}%)")
print("- Transfer learning reuses features learned from a large dataset, which supports better "
      "performance and faster convergence when task-specific data is limited.")

### Future Improvements
- Use the full dataset (larger `MAX_PER_CLASS`) and a higher input resolution such as 224 x 224.
- Compare other pretrained architectures such as ResNet50, EfficientNet and VGG16.
- Evaluate on real field images with complex backgrounds, since PlantVillage images have plain backgrounds.
- Address class imbalance using class weights or stronger augmentation.
- Convert the model to TensorFlow Lite and deploy it in a mobile application for farmers.